# Aggregation MD directory Analysis Notebook

This notebook provides a structured workflow for analyzing **aggregation MD directory** with various `Temperatures` and `Deposition Interval times`, using separate Python modules.

## Workflow Overview:
1. **Load and extract Data**: 

---
## Part 1: Load data & Read variables
- Read variables and attributes from specific file

In [ ]:
import data_extraction
import visualization
import system_analysis

output_dir = "/home/hadis/custom_vector/runs/particleOriented/march15_aggregationRun/outputs"

In [ ]:
visualization.plot_configuration_aggregation(output_dir, .15, 3, step_target=-1)

In [ ]:
visualization.animate_position_aggregation(output_dir, temperature=0.15, deposition_rate=0.5, frame_skip=1)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import data_extraction

def extract_order_parameter(output_dir, temperatures, deposition_rates):
    ranges = data_extraction.extract_ranges(output_dir)

    temperature_numbers = len(temperatures)
    deposition_rates_numbers = len(deposition_rates)
    order_matrix = np.zeros((temperature_numbers, deposition_rates_numbers))

    for i, temperature in enumerate(temperatures):
        for j, deposition_rate in enumerate(deposition_rates):
            formatted_temp = f"{temperature:.2f}"
            formatted_dep_rate = f"{deposition_rate:.1f}"

            file_path = os.path.join(output_dir, f'aggregated_{formatted_temp}_{formatted_dep_rate}.bp')

            order_value = data_extraction.read_variable(file_path, 'orientational order')['data']

            order_matrix[i, j] = order_value[-1]

            print(f"File {i+1}/{temperature_numbers}, {j+1}/{deposition_rates_numbers} stored!                 ", end='\r')

    return order_matrix

def plot_parameter_heatmap(parameter_matrix, parameter_name, temperatures, deposition_rates, cmap='RdYlBu'):
    fig, ax = plt.subplots(figsize=(8, 6))
    
    c = ax.contourf(deposition_rates, temperatures, parameter_matrix, levels=100, cmap=cmap)

    fig.colorbar(c, ax=ax, label=parameter_name)
    ax.set_title(f'{parameter_name} Heatmap')
    ax.set_xlabel('Deposition Rate')
    ax.set_ylabel('Temperature')

    plt.show()


temperatures = list(np.arange(0.05, 0.35, 0.05))
deposition_rates = list(np.arange(0.5, 7, 0.5))

order_matrix = extract_order_parameter(output_dir, temperatures, deposition_rates)
plot_parameter_heatmap(order_matrix, 'Orientational Order', temperatures, deposition_rates)


In [ ]:
import os
import numpy as np
import data_extraction

def calculate_gyration_radius(positions_data):
    positions_xy = positions_data[:, :, :2]  # Ignore phi
    COM = np.mean(positions_xy, axis=1)
    Rg = []

    for step in range(positions_xy.shape[0]):
        squared_distance = np.sum((positions_xy[step] - COM[step])**2, axis=1)
        Rg.append(np.sqrt(np.mean(squared_distance)))
        
    return np.array(Rg)

def compute_radius_matrix(output_dir, temperatures, deposition_rates):
    temperature_numbers = len(temperatures)
    deposition_rates_numbers = len(deposition_rates)
    
    radius_matrix = np.zeros((temperature_numbers, deposition_rates_numbers))

    for i, temperature in enumerate(temperatures):
        for j, deposition_rate in enumerate(deposition_rates):
            
            formatted_temp = f"{temperature:.2f}"
            formatted_dep_rate = f"{deposition_rate:.1f}"
                        
            file_path = os.path.join(output_dir, f'aggregated_{formatted_temp}_{formatted_dep_rate}.bp')
            
            positions_data = data_extraction.read_variable(file_path, 'positions', print_message=False)['data']
            radius_data = calculate_gyration_radius(positions_data)
            # radius_value = np.mean(radius_data)
            radius_value = radius_data[-1]

            radius_matrix[i, j] = radius_value
            print(f"File {i+1}/{temperature_numbers}, {j+1}/{deposition_rates_numbers} stored!                 ", end='\r')

    return radius_matrix

RG_matrix = compute_radius_matrix(output_dir, temperatures, deposition_rates)


In [ ]:
plot_parameter_heatmap(RG_matrix, 'Radius of Gyration', temperatures, deposition_rates, cmap='coolwarm')